## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr


/Users/tarkshya/Work/LLM Engineering/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
# The most predictable and deterministic Token is picked up with temp = 0.
# Whereas higher Temp means we occasionally pick tokens which didn't get the top priority but maybe got the second top or something like that.

# In case of OpenAI temp varies from 0 to 2 and a temp of 1 means that if a token were given 10% prob, then you would pick that one 10% of the time. so basically 10 in 100 times that token would come.( could be creative or a bit odd sometimes).
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME,embedding_function=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6370.56it/s]


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
# Stored all the docx in chunks in the retriever . Think of it like a whole library of information
retriever = vectorstore.as_retriever()

In [7]:
docs = retriever.invoke("who is avery")

In [22]:

context = "\n\n".join(doc.page_content for doc in docs)

print(context)

## Other HR Notes
- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  
- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  
- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.
- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  

Avery Lancaster has demonstrated resilience and adaptability throughout her career at Insurellm, positioning the company as a key player in the insurance technology landscape.

## Other HR Notes
- **Professional Devel

More precisely, join() puts "\n\n" between each retrieved chunk:

Chunk 1
\n\n
Chunk 2
\n\n
Chunk 3

So it doesn't add two newlines after the final chunk—it uses them as the separator between chunks.

In [ ]:
llm = ChatOpenAI(model=MODEL,temperature=0)

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)

In [ ]:
question = "Who is Avery?"
response = llm.invoke([SystemMessage(content=system_prompt),HumanMessage(content=question)])
response.content

In [ ]:
def answer_question(question:str,history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt),HumanMessage(content=question)])
    return response.content
